In [1]:
import pymysql
import pandas as pd
from typing import Optional, List

def get_price_stock_from_db(
    db_info: dict,
    tickers: Optional[List[str]] = None,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    format: str = "long"  # "long" 또는 "wide"
) -> pd.DataFrame:
    """
    us_required_return_result 테이블에서 price_stock 데이터 추출

    Parameters:
    -----------
    db_info : dict
        DB 연결 정보 (host, port, user, password, database)
    tickers : list, optional
        추출할 ticker 리스트. None이면 전체
    start_date : str, optional
        시작 날짜 (YYYY-MM-DD)
    end_date : str, optional
        종료 날짜 (YYYY-MM-DD)
    format : str
        "long" - (date, ticker, value) 형태
        "wide" - ticker를 컬럼으로 pivot한 형태

    Returns:
    --------
    pd.DataFrame
    """

    # SQL 쿼리 작성
    where_clauses = ["indicator = 'price_stock'"]
    params = []

    if tickers is not None and len(tickers) > 0:
        placeholders = ", ".join(["%s"] * len(tickers))
        where_clauses.append(f"ticker IN ({placeholders})")
        params.extend(tickers)

    if start_date is not None:
        where_clauses.append("date >= %s")
        params.append(start_date)

    if end_date is not None:
        where_clauses.append("date <= %s")
        params.append(end_date)

    where_sql = " AND ".join(where_clauses)

    sql = f"""
    SELECT date, ticker, value as price
    FROM us_required_return_result
    WHERE {where_sql}
    ORDER BY date, ticker;
    """

    # DB 연결 및 데이터 추출
    conn = pymysql.connect(
        host=db_info['host'],
        port=db_info.get('port', 3307),
        user=db_info['user'],
        password=db_info['password'],
        db=db_info.get('database', 'investar'),
        charset='utf8mb4'
    )

    try:
        df = pd.read_sql(sql, conn, params=params)
    finally:
        conn.close()

    if df.empty:
        print(f"[WARNING] No price_stock data found")
        return df

    # 날짜 형식 변환
    df['date'] = pd.to_datetime(df['date'])
    df['price'] = pd.to_numeric(df['price'], errors='coerce')

    # NaN 제거
    df = df.dropna(subset=['price'])

    print(f"[INFO] Extracted {len(df)} rows from {df['ticker'].nunique()} tickers")
    print(f"[INFO] Date range: {df['date'].min()} to {df['date'].max()}")

    # Wide format으로 변환 (요청 시)
    if format == "wide":
        df = df.pivot(index='date', columns='ticker', values='price')
        df = df.sort_index()

    return df


def get_price_stock_summary(db_info: dict) -> pd.DataFrame:
    """
    price_stock 데이터 요약 정보 조회

    Returns:
    --------
    pd.DataFrame: ticker별 데이터 개수, 시작일, 종료일
    """
    sql = """
    SELECT
        ticker,
        COUNT(*) as data_count,
        MIN(date) as first_date,
        MAX(date) as last_date,
        DATEDIFF(MAX(date), MIN(date)) as date_range_days
    FROM us_required_return_result
    WHERE indicator = 'price_stock'
    GROUP BY ticker
    ORDER BY data_count DESC;
    """

    conn = pymysql.connect(
        host=db_info['host'],
        port=db_info.get('port', 3307),
        user=db_info['user'],
        password=db_info['password'],
        db=db_info.get('database', 'investar'),
        charset='utf8mb4'
    )

    try:
        df = pd.read_sql(sql, conn)
    finally:
        conn.close()

    print(f"[INFO] Total tickers with price_stock data: {len(df)}")

    return df


def check_missing_tickers(
    db_info: dict,
    target_tickers: List[str]
) -> tuple:
    """
    price_stock 데이터가 있는 ticker와 없는 ticker 구분

    Returns:
    --------
    tuple: (존재하는 ticker 리스트, 없는 ticker 리스트)
    """
    sql = """
    SELECT DISTINCT ticker
    FROM us_required_return_result
    WHERE indicator = 'price_stock';
    """

    conn = pymysql.connect(
        host=db_info['host'],
        port=db_info.get('port', 3307),
        user=db_info['user'],
        password=db_info['password'],
        db=db_info.get('database', 'investar'),
        charset='utf8mb4'
    )

    try:
        df = pd.read_sql(sql, conn)
    finally:
        conn.close()

    existing_tickers = set(df['ticker'].tolist())
    target_set = set(target_tickers)

    found = sorted(list(existing_tickers & target_set))
    missing = sorted(list(target_set - existing_tickers))

    print(f"[INFO] Found: {len(found)} tickers")
    print(f"[INFO] Missing: {len(missing)} tickers")

    return found, missing


# ============================================
# 사용 예시
# ============================================

# if __name__ == "__main__":
#     # DB 설정
#     db_info = {
#         'host': 'localhost',  # 또는 실제 호스트
#         'port': 3307,
#         'user': 'your_user',
#         'password': 'your_password',
#         'database': 'investar'
#     }
#
#     # 1. 전체 price_stock 데이터 요약
#     print("=" * 80)
#     print("1. Price Stock Data Summary")
#     print("=" * 80)
#     summary_df = get_price_stock_summary(db_info)
#     print(summary_df.head(20))
#
#     # 2. 특정 ticker들의 price_stock 데이터 추출 (Long format)
#     print("\n" + "=" * 80)
#     print("2. Extract Specific Tickers (Long Format)")
#     print("=" * 80)
#     tickers = ['AAPL', 'MSFT', 'NVDA', 'SPY']
#     price_df = get_price_stock_from_db(
#         db_info=db_info,
#         tickers=tickers,
#         start_date='2020-01-01',
#         format='long'
#     )
#     print(price_df.head(10))
#     print(price_df.tail(10))
#
#     # 3. Wide format으로 추출 (날짜별로 ticker가 컬럼)
#     print("\n" + "=" * 80)
#     print("3. Extract in Wide Format")
#     print("=" * 80)
#     price_wide_df = get_price_stock_from_db(
#         db_info=db_info,
#         tickers=tickers,
#         start_date='2020-01-01',
#         format='wide'
#     )
#     print(price_wide_df.head())
#
#     # 4. 특정 ticker 리스트에서 누락된 데이터 확인
#     print("\n" + "=" * 80)
#     print("4. Check Missing Tickers")
#     print("=" * 80)
#     from DATA.us_target_ticker_list import ticker_list
#     found, missing = check_missing_tickers(db_info, ticker_list[:100])
#     print(f"\nFirst 10 found tickers: {found[:10]}")
#     print(f"First 10 missing tickers: {missing[:10]}")
#
#     # 5. 특정 ticker의 최근 데이터만 추출
#     print("\n" + "=" * 80)
#     print("5. Recent Data for SPY")
#     print("=" * 80)
#     spy_recent = get_price_stock_from_db(
#         db_info=db_info,
#         tickers=['SPY'],
#         start_date='2026-01-01',
#         format='long'
#     )
#     print(spy_recent)

In [12]:
# DB 정보 설정
from DATA.stock_invest_function import get_db_host

db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",   # 실제 비밀번호
    "database": "investar",
}

# 요약 정보 확인
summary = get_price_stock_summary(db_info)
print(summary.head())

# 특정 ticker 데이터 추출
df = get_price_stock_from_db(
    db_info=db_info,
    tickers=['ANET'],
    start_date='2015-01-01'
)
print(df)

[INFO] Total tickers with price_stock data: 2213
  ticker  data_count  first_date   last_date  date_range_days
0   EXPE        2783  2014-12-31  2026-01-26             4044
1   GILD        2783  2014-12-31  2026-01-26             4044
2   HWKN        2783  2014-12-31  2026-01-26             4044
3   JBLU        2783  2014-12-31  2026-01-26             4044
4   LRCX        2783  2014-12-31  2026-01-26             4044
[WARNING] No price_stock data found
Empty DataFrame
Columns: [date, ticker, price]
Index: []


In [14]:
summary

,ticker,data_count,first_date,last_date,date_range_days
0,EXPE,2783,2014-12-31,2026-01-26,4044
1,GILD,2783,2014-12-31,2026-01-26,4044
2,HWKN,2783,2014-12-31,2026-01-26,4044
3,JBLU,2783,2014-12-31,2026-01-26,4044
4,LRCX,2783,2014-12-31,2026-01-26,4044
...,...,...,...,...,...
2208,WSTN,8,2025-12-31,2026-01-12,12
2209,SORNU,5,2026-01-06,2026-01-12,6
2210,ARTCU,5,2026-01-06,2026-01-12,6
2211,BBCQU,3,2026-01-08,2026-01-12,4
